# Interview statworx

In [1]:
import os
import uuid

import gradio as gr
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document



/Users/susannazirizadeh/Documents/Private/Bewerbung/statworx/use_case/statworx_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:


load_dotenv()


True

In [12]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("Missing OPENROUTER_API_KEY. Set it in env or .env")

MODEL_NAME = "openai/gpt-4o-mini"  # change to any OpenRouter model you prefer

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
    default_headers={
        "HTTP-Referer": os.getenv("OPENROUTER_SITE_URL", "http://localhost"),
        "X-Title": os.getenv("OPENROUTER_APP_NAME", "LangChain Chatbot"),
    },
)

In [4]:
docs = [
    Document(page_content="Our refund policy allows returns within 30 days with a receipt."),
    Document(page_content="To reset your password, click 'Forgot Password' on the login page."),
    Document(page_content="Support is available Monday to Friday from 9:00 to 17:00 CET."),
]
print(f"Loaded {len(docs)} sample documents")

Loaded 3 sample documents


In [5]:

#docs = []
# for path in glob.glob("docs/*.txt"):
#    with open(path, "r", encoding="utf-8") as f:
#        docs.append(Document(page_content=f.read(), metadata={"source": path}))

#print(f"Loaded {len(docs)} documents from docs/")


In [6]:
# 1) Chunk documents
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(docs)

# 2) Local embeddings model (no OpenRouter needed for embeddings)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3) Build Chroma vector store (persisted)
PERSIST_DIR = "../chroma_db"
COLLECTION_NAME = "my_docs"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

# Persist to disk (important for re-use across sessions)
vectorstore.persist()

# 4) Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Indexed {len(chunks)} chunks into ChromaDB at {PERSIST_DIR}")

/var/folders/7g/05v9_xk54r9492td2ywlkd2r0000gn/T/ipykernel_91236/90350432.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8008.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 3 chunks into ChromaDB at ../chroma_db


/var/folders/7g/05v9_xk54r9492td2ywlkd2r0000gn/T/ipykernel_91236/90350432.py:20: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [7]:
PERSIST_DIR = "../chroma_db"
COLLECTION_NAME = "my_docs"

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("ChromaDB loaded from disk ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13550.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB loaded from disk ✅


/var/folders/7g/05v9_xk54r9492td2ywlkd2r0000gn/T/ipykernel_91236/3383319597.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [13]:

SYSTEM_PROMPT = """You are a helpful assistant.
Be concise, accurate, and ask clarifying questions when needed.
If context is provided, use it. If not, answer normally.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()  # ✅ always outputs a string

In [9]:

_STORE = {}

def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in _STORE:
        _STORE[session_id] = InMemoryChatMessageHistory()
    return _STORE[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history=get_history,
    input_messages_key="input",
    history_messages_key="history",
)


def retrieve_context(query: str) -> str:
    docs = retriever.invoke(query)
    return "\n\n".join([f"- {d.page_content}" for d in docs])

def chat_fn(message: str, history, *args):
    use_rag = args[0] if len(args) > 0 else True
    session_id = args[1] if len(args) > 1 else "default-session"

    user_input = message
    if use_rag:
        ctx = retrieve_context(message).strip()
        if ctx:
            user_input = f"Context:\n{ctx}\n\nUser question: {message}"

    answer_text = chain_with_memory.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    )

    return answer_text  # ✅ already a string



In [14]:


INITIAL_CHAT = [
    {"role": "assistant", "content": "Hey 👋 How can I help you today?"}
]

with gr.Blocks() as demo:
    use_rag = gr.Checkbox(value=True, label="Use RAG (ChromaDB retrieval)")
    session_id = gr.State(str(uuid.uuid4()))

    chatbot = gr.Chatbot(value=INITIAL_CHAT)

    gr.ChatInterface(
        fn=chat_fn,
        chatbot=chatbot,
        additional_inputs=[use_rag, session_id],
        title="🤖 LangChain + OpenRouter Chatbot (Optional RAG)",
        description="Toggle RAG to answer using your local ChromaDB documents."
    )

demo.launch()



* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/susannazirizadeh/Documents/Private/Bewerbung/statworx/use_case/statworx_env/lib/python3.13/site-packages/gradio/queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/susannazirizadeh/Documents/Private/Bewerbung/statworx/use_case/statworx_env/lib/python3.13/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Users/susannazirizadeh/Documents/Private/Bewerbung/statworx/use_case/statworx_env/lib/python3.13/site-packages/gradio/blocks.py", line 2162, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Users/susannazirizadeh/Documents/Private/Bewerbung/statworx/use_case/stat